In [3]:
# Importación de librerías esenciales
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# Modificación para importar los modelos fuera de la carpeta actual
import os
import sys

root_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
if root_dir not in sys.path:
    sys.path.append(root_dir)

# Importación del modelo
from src.LogisticRegression import LogisticRegression
from src.utils_and_metrics import train_test_split, accuracy, confusion_matrix, recall, f1_score, precision

#### **1. Preparación de Datos: Estandarización y Binarización One-vs-Rest**
A diferencia de la exclusión binaria, aquí utilizaremos todo el conjunto de datos (incluyendo la Clase 2). Para adaptar nuestro modelo binario a un problema de 3 clases, crearemos **3 vectores objetivo independientes**:
* **y_train_c1:** Positivo (1) si es Normal, Negativo (0) para el resto.
* **y_train_c2:** Positivo (1) si es Sospechoso, Negativo (0) para el resto.
* **y_train_c3:** Positivo (1) si es Patológico, Negativo (0) para el resto.

In [4]:
#Cargar el dataset
df = pd.read_csv('../data/processed/CTG_clean.csv')

# Separación de características y variable objetivo
X_todo = df.drop(columns=['NSP']).to_numpy()
y_todo_og = df['NSP'].to_numpy().astype(int)

# Partición de datos (80% entrenamiento, 20% prueba)
# Se utiliza la función custom_train_test_split del curso
X_train_raw, X_test_raw, y_train_og, y_test_og = train_test_split(
    X_todo, y_todo_og, test_size=0.20, seed=42
)

# Estandarización Z-Score (usando solo X_train para evitar data leakage)
mean = X_train_raw.mean(axis=0)
std = X_train_raw.std(axis=0)
std = np.where(std == 0, 1.0, std)

X_train = (X_train_raw - mean) / std
X_test = (X_test_raw - mean) / std

# Binarización One-vs-Rest (OvR) para el entrenamiento
y_train_c1 = np.where(y_train_og == 1, 1, 0)
y_train_c2 = np.where(y_train_og == 2, 1, 0)
y_train_c3 = np.where(y_train_og == 3, 1, 0)

print(f"Dimensiones de X_train estandarizado: {X_train.shape}")
print(f"Distribución Clase 1 vs Resto: {np.bincount(y_train_c1)}")
print(f"Distribución Clase 2 vs Resto: {np.bincount(y_train_c2)}")
print(f"Distribución Clase 3 vs Resto: {np.bincount(y_train_c3)}")

Dimensiones de X_train estandarizado: (1700, 21)
Distribución Clase 1 vs Resto: [ 380 1320]
Distribución Clase 2 vs Resto: [1470  230]
Distribución Clase 3 vs Resto: [1550  150]


#### **2. Entrenamiento del Ensamble One-vs-Rest**
Instanciamos y entrenamos tres modelos de Regresión Logística de manera independiente. Cada modelo se especializará en detectar su propia clase en contra de todas las demás.


In [5]:
# Instanciamos los 3 modelos con los mismos hiperparámetros
model_c1 = LogisticRegression(learning_rate=0.1, n_iterations=2000)
model_c2 = LogisticRegression(learning_rate=0.1, n_iterations=2000)
model_c3 = LogisticRegression(learning_rate=0.1, n_iterations=2000)

print("Entrenando Modelo 1 (Especialista en detectar Normales)...")
model_c1.fit(X_train, y_train_c1)

print("Entrenando Modelo 2 (Especialista en detectar Sospechosos)...")
model_c2.fit(X_train, y_train_c2)

print("Entrenando Modelo 3 (Especialista en detectar Patológicos)...")
model_c3.fit(X_train, y_train_c3)

print("Entrenamiento multiclase completado con éxito.")

Entrenando Modelo 1 (Especialista en detectar Normales)...
Entrenando Modelo 2 (Especialista en detectar Sospechosos)...
Entrenando Modelo 3 (Especialista en detectar Patológicos)...
Entrenamiento multiclase completado con éxito.


#### **3. Evaluación y Decisión Multiclase (Argmax)**
Para predecir el diagnóstico de un paciente nuevo en el conjunto de prueba, le consultamos a los 3 modelos la probabilidad de que pertenezca a su clase especialista. El diagnóstico final se asigna a la clase que arroje **la probabilidad más alta** (`np.argmax`).


In [6]:
# 1. Obtener las 3 probabilidades predichas por la función sigmoide para cada paciente
probs_c1 = model_c1.predict_proba(X_test)
probs_c2 = model_c2.predict_proba(X_test)
probs_c3 = model_c3.predict_proba(X_test)

# 2. Agrupar probabilidades en una matriz de (n_samples, 3)
probs_matrix = np.column_stack((probs_c1, probs_c2, probs_c3))

# 3. Decisión Multiclase: np.argmax devuelve el índice con mayor probabilidad (0, 1 o 2)
# Le sumamos 1 para regresar a la escala clínica original (1, 2 o 3)
prediction_prueba = np.argmax(probs_matrix, axis=1) + 1

# 4. Cálculo de métricas
n_classes = 3
# Ajustamos a índice base cero (0,1,2) para la función custom confusion_matrix
matriz_test = confusion_matrix(y_test_og - 1, prediction_prueba - 1, n_classes)
acc_test = accuracy(y_test_og, prediction_prueba)
precisions_test = precision(matriz_test)
recalls_test = recall(matriz_test)
f1s_test = f1_score(matriz_test)

# 5. Impresión de Evaluación Final
print("\n--- Resultados de Rendimiento en Prueba (OvR) ---")
print(f"Exactitud Global (Accuracy): {acc_test * 100:.2f}%\n")

print("--- Métricas detalladas por Clase ---")
nombres_clases = ["1 (Normal)", "2 (Sospechoso)", "3 (Patológico)"]
for i, nombre in enumerate(nombres_clases):
    print(f"Clase {nombre:14s} -> Precision: {precisions_test[i]*100:.2f}% | Recall: {recalls_test[i]*100:.2f}% | F1-Score: {f1s_test[i]*100:.2f}%")

print(f"\nPromedio Macro F1-Score: {np.mean(f1s_test)*100:.2f}%")
print("\nMatriz de Confusión (Filas: Real [1,2,3] | Columnas: Predicho [1,2,3]):")
print(matriz_test)


--- Resultados de Rendimiento en Prueba (OvR) ---
Exactitud Global (Accuracy): 88.97%

--- Métricas detalladas por Clase ---
Clase 1 (Normal)     -> Precision: 93.29% | Recall: 95.52% | F1-Score: 94.40%
Clase 2 (Sospechoso) -> Precision: 66.13% | Recall: 63.08% | F1-Score: 64.57%
Clase 3 (Patológico) -> Precision: 85.71% | Recall: 69.23% | F1-Score: 76.60%

Promedio Macro F1-Score: 78.52%

Matriz de Confusión (Filas: Real [1,2,3] | Columnas: Predicho [1,2,3]):
[[320  14   1]
 [ 22  41   2]
 [  1   7  18]]
